# FLUX v2 — Phase 3: Wave Generator
**First Words — GRU-based next-wave predictor with decode loss from step 1**

Prerequisites: `phase1_v2.phase.pt` and `phase2_v2.phase.pt` must exist on `UnseenGAP/FLUX` HuggingFace Hub.

| Cell | Name | Run When |
|------|------|----------|
| 1 | SETUP & CLONE | **First time only** — installs deps, clones `v2` branch |
| 2 | REFRESH | **Before every run** — clears state, pulls latest, wipes results |
| 3 | HARDWARE & CREDENTIALS | After Refresh — resolves tokens, downloads Phase 1 & 2 checkpoints |
| 4 | SMOKE TEST | After Hardware — verifies imports and shapes |
| 5 | TRAINING | Core training loop |
| 6 | TRAINING DIAGNOSTICS | Immediately after training — catch show-stoppers |
| 7 | UPLOAD TO HUGGINGFACE | After passing diagnostics |
| 8 | TEST 1 — loads from HF | Decode gate accuracy |
| 9 | TEST 2 — loads from HF | Context sensitivity |
| 10 | TEST 3 — loads from HF | Valid word rate |
| 11 | DEMO 1 — loads from HF | Generation pipeline |
| 12 | DEMO 2 — loads from HF | Context cluster heatmap |
| 13 | SAVE RESULTS | Logs + graphs → `V2_results/phase3/`, push to GitHub `v2` branch |
| 14 | FINAL SUMMARY | Results summary |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — SETUP & CLONE  (run once per new Colab/Kaggle session)   ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys

# ── Detect runtime ────────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    RUNTIME  = 'kaggle'
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    RUNTIME  = 'colab'
    WORK_DIR = '/content'
else:
    RUNTIME  = 'local'
    WORK_DIR = os.path.expanduser('~')

REPO_PATH   = f'{WORK_DIR}/FLUX'
GITHUB_REPO = 'https://github.com/Unseengap/FLUX.git'
BRANCH      = 'v2'

print(f'Runtime  : {RUNTIME}')
print(f'WORK_DIR : {WORK_DIR}')

# ── Clone or update repo ──────────────────────────────────────────────
os.chdir(WORK_DIR)
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_REPO], check=True)
    print(f'Repo cloned (branch: {BRANCH})')
else:
    subprocess.run(['git', '-C', REPO_PATH, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_PATH, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_PATH, 'pull', 'origin', BRANCH], check=True)
    print(f'Repo updated (branch: {BRANCH})')

os.chdir(REPO_PATH)

# ── Install pip dependencies ──────────────────────────────────────────
# setup.py is a custom directory-init script, NOT a setuptools package.
# Never run `pip install -e .` — run it directly as a Python script.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
     'torch', 'huggingface_hub', 'datasets', 'transformers',
     'evaluate', 'rich', 'tqdm', 'matplotlib', 'scipy'],
    check=True,
)
print('pip dependencies installed')

# ── Run custom setup.py to create project directories ─────────────────
subprocess.run([sys.executable, 'setup.py'], check=True)
print('Project structure initialised')

# ── Add repo root to sys.path so all imports resolve ──────────────────
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f'sys.path: {REPO_PATH} added')
print(f'CWD: {os.getcwd()}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — REFRESH  (run before EVERY session, EVERY debug cycle)   ║
# ╚══════════════════════════════════════════════════════════════════════╝
%reset -f

# ── Re-define all constants ───────────────────────────────────────────
import os, gc, shutil, subprocess

if os.path.exists('/kaggle/working'):
    RUNTIME  = 'kaggle'
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    RUNTIME  = 'colab'
    WORK_DIR = '/content'
else:
    RUNTIME  = 'local'
    WORK_DIR = os.path.expanduser('~')

REPO_PATH      = f'{WORK_DIR}/FLUX'
RESULTS_DIR    = f'{REPO_PATH}/v2/V2_results/phase3'
CHECKPOINT_DIR = f'{REPO_PATH}/checkpoints'
BRANCH         = 'v2'
HF_USER        = 'UnseenGAP'
HF_REPO_ID     = 'UnseenGAP/FLUX'
GITHUB_REPO    = 'https://github.com/Unseengap/FLUX.git'

# Tokens will be properly resolved in Cell 3;
# pull from env here so Cell 3 can use the env-var path first
HF_TOKEN     = os.environ.get('HF_TOKEN', '')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')

# ── Free GPU VRAM & CPU RAM ───────────────────────────────────────────
try:
    import torch
    torch.cuda.empty_cache()
    print('  ✓ GPU VRAM cleared')
except Exception:
    pass
gc.collect()
print('  ✓ CPU RAM collected')

# ── Pull latest code ──────────────────────────────────────────────────
if os.path.exists(REPO_PATH):
    subprocess.run(['git', '-C', REPO_PATH, 'pull', 'origin', BRANCH], check=True)
    print(f'  ✓ git pull origin {BRANCH}')
else:
    print(f'  ⚠ REPO_PATH not found — run Cell 1 first')

# ── Wipe and recreate RESULTS_DIR ────────────────────────────────────
if os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
os.makedirs(f'{RESULTS_DIR}/logs',   exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/graphs', exist_ok=True)
print(f'  ✓ RESULTS_DIR wiped and recreated: {RESULTS_DIR}')

# ── Verify key files exist ────────────────────────────────────────────
V2_P3 = f'{REPO_PATH}/v2/phase3'
critical = [
    f'{V2_P3}/wave_generator.py',
    f'{V2_P3}/train_generator.py',
    f'{V2_P3}/test_phase3_test1.py',
    f'{V2_P3}/test_phase3_test2.py',
    f'{V2_P3}/test_phase3_test3.py',
    f'{V2_P3}/demo_phase3_demo1.py',
    f'{V2_P3}/demo_phase3_demo2.py',
]
all_ok = True
for f in critical:
    exists = os.path.exists(f)
    status = '✓' if exists else '✗'
    print(f'  {status} {os.path.basename(f)}')
    if not exists:
        all_ok = False

# ── Summary ───────────────────────────────────────────────────────────
print()
print('── REFRESH COMPLETE ──────────────────────────────────────────')
print(f'  Runtime      : {RUNTIME}')
print(f'  REPO_PATH    : {REPO_PATH}')
print(f'  RESULTS_DIR  : {RESULTS_DIR}')
print(f'  HF_TOKEN     : {"SET (len=" + str(len(HF_TOKEN)) + ")" if HF_TOKEN else "NOT SET — resolve in Cell 3"}')
print(f'  GITHUB_TOKEN : {"SET (len=" + str(len(GITHUB_TOKEN)) + ")" if GITHUB_TOKEN else "NOT SET — resolve in Cell 3"}')
print(f'  Files OK     : {all_ok}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — HARDWARE & CREDENTIALS                                   ║
# ╚══════════════════════════════════════════════════════════════════════╝
import sys, os, torch
from pathlib import Path

sys.path.insert(0, REPO_PATH)
from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=3)
log.separator('Phase 3 v2 — Wave Generator')

# ── Hardware ──────────────────────────────────────────────────────────
DEVICE = get_device()
log.info(f'Runtime  : {RUNTIME}')
log.info(f'Device   : {DEVICE}')
if torch.cuda.is_available():
    gpu        = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram  = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated()) / 1e9
    log.success(f'GPU: {gpu}')
    log.metric('Total VRAM', f'{total_vram:.1f} GB')
    log.metric('Free VRAM',  f'{free_vram:.1f} GB')
    if total_vram < 8:
        log.warning('< 8 GB VRAM — training will be slow')

# ── Token resolution: env → Colab → Kaggle ────────────────────────────
# Priority order mandated by NOTEBOOKS_SPEC.md
def _resolve_token(name: str) -> str:
    t = os.environ.get(name, '').strip()
    if t:
        return t
    try:
        from google.colab import userdata
        t = (userdata.get(name) or '').strip()
        if t:
            return t
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        t = (UserSecretsClient().get_secret(name) or '').strip()
        if t:
            return t
    except Exception:
        pass
    return ''

HF_TOKEN     = _resolve_token('HF_TOKEN')
GITHUB_TOKEN = _resolve_token('GITHUB_TOKEN')

# Write back to env so Cell 7 re-resolution can find them
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

if HF_TOKEN:
    log.success(f'HF_TOKEN resolved (len={len(HF_TOKEN)})')
else:
    log.warning('HF_TOKEN not found — add via Colab/Kaggle secrets')

if GITHUB_TOKEN:
    log.success(f'GITHUB_TOKEN resolved (len={len(GITHUB_TOKEN)})')
else:
    log.warning('GITHUB_TOKEN not found — Cell 13 GitHub push will be skipped')

# ── HuggingFace login ────────────────────────────────────────────────
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        log.success('HuggingFace login OK')
    except Exception as e:
        log.warning(f'HuggingFace login failed: {e}')

# ── Download prior phase checkpoints ─────────────────────────────────
# Phase 3 depends on phase1_v2.phase.pt AND phase2_v2.phase.pt
from huggingface_hub import hf_hub_download
import shutil

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for hf_filename, local_name in [
    ('v2/phase1_v2.phase.pt', 'phase1_v2.phase.pt'),
    ('v2/phase2_v2.phase.pt', 'phase2_v2.phase.pt'),
]:
    local_path = os.path.join(CHECKPOINT_DIR, local_name)
    if not os.path.exists(local_path):
        if HF_TOKEN:
            try:
                cached = hf_hub_download(
                    repo_id=HF_REPO_ID,
                    filename=hf_filename,
                    token=HF_TOKEN,
                )
                shutil.copy2(cached, local_path)
                size_mb = os.path.getsize(local_path) / 1e6
                log.success(f'Downloaded {local_name} ({size_mb:.1f} MB)')
            except Exception as e:
                log.warning(f'Could not download {local_name}: {e}')
        else:
            log.warning(f'{local_name} missing and no HF_TOKEN to download')
    else:
        size_mb = os.path.getsize(local_path) / 1e6
        log.success(f'{local_name} already present ({size_mb:.1f} MB)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — SMOKE TEST                                               ║
# ╚══════════════════════════════════════════════════════════════════════╝
import sys, torch
from pathlib import Path

V2_PHASE1 = f'{REPO_PATH}/v2/phase1'
V2_PHASE2 = f'{REPO_PATH}/v2/phase2'
V2_PHASE3 = f'{REPO_PATH}/v2/phase3'

for p in [V2_PHASE1, V2_PHASE2, V2_PHASE3, REPO_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

from cse            import ContinuousSemanticEncoder
from wave_to_field  import WaveToField
from wave_generator import WaveGenerator

# ContinuousSemanticEncoder takes wave_dims (dict), not wave_dim (int) — use defaults
cse = ContinuousSemanticEncoder()
w2f = WaveToField(wave_dim=432, field_dim=512)
gen = WaveGenerator(wave_dim=432, field_features=512)

# Verify phase 1 & 2 checkpoints load
p1_ckpt = torch.load(f'{CHECKPOINT_DIR}/phase1_v2.phase.pt', map_location='cpu')
p2_ckpt = torch.load(f'{CHECKPOINT_DIR}/phase2_v2.phase.pt', map_location='cpu')
assert 'state_dict' in p1_ckpt, 'phase1 checkpoint missing state_dict'
assert 'state_dict' in p2_ckpt, 'phase2 checkpoint missing state_dict'
log.success('Phase 1 & 2 checkpoints load OK')

# Verify shapes
with torch.no_grad():
    wave     = cse.encode('Hello world')
    ctx      = w2f(wave.full.mean(dim=0))
    waves, _ = gen.generate(field_context=ctx, max_waves=5)

assert wave.full.shape[1] == 432, f'Bad wave shape: {wave.full.shape}'
assert ctx.shape[0] == 512,       f'Bad ctx shape: {ctx.shape}'
assert waves.shape[1] == 432,     f'Bad generated shape: {waves.shape}'

# Verify WaveGenerator has trainable params
n_params = sum(p.numel() for p in gen.parameters() if p.requires_grad)
assert n_params > 0, 'WaveGenerator has no trainable parameters'

log.success(f'SMOKE TEST PASSED — WaveGenerator has {n_params:,} trainable parameters')
log.success(f'Generated wave shape: {list(waves.shape)}')

del cse, w2f, gen, p1_ckpt, p2_ckpt, wave, ctx, waves
torch.cuda.empty_cache()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — TRAINING                                                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
import sys, importlib, os
from pathlib import Path

if V2_PHASE3 not in sys.path:
    sys.path.insert(0, V2_PHASE3)

# ── Force-reload to prevent stale bytecode from failed prior run ──────
import train_generator as _tg_module
importlib.reload(_tg_module)
from train_generator import train, PHASE3_CONFIG

log.separator('Cell 5 — Training')
log.info(f'Steps               : {PHASE3_CONFIG["steps"]:,}')
log.info(f'Learning rate       : {PHASE3_CONFIG["lr"]}')
log.info(f'GRU hidden          : {PHASE3_CONFIG["gru_hidden"]}')
log.info(f'SS ramp             : {PHASE3_CONFIG["ss_start"]} → {PHASE3_CONFIG["ss_end"]}')
log.info(f'wave_loss_weight    : {PHASE3_CONFIG["wave_loss_weight"]}  (regulariser only — was dominant)')
log.info(f'decode_loss_weight  : {PHASE3_CONFIG["decode_loss_weight"]}  (bridge path — differentiable signal)')
log.info(f'contrastive_weight  : {PHASE3_CONFIG["contrastive_weight"]}')
log.info(f'reinforce_weight    : {PHASE3_CONFIG["reinforce_weight"]}  (REINFORCE policy gradient)')
log.info(f'reinforce_interval  : {PHASE3_CONFIG["reinforce_interval"]}  steps')
log.info(f'reinforce_batch     : {PHASE3_CONFIG["reinforce_batch"]}  rollouts per RL update')
log.info(f'reinforce_sigma     : {PHASE3_CONFIG["reinforce_sigma"]}  exploration noise')
log.info(f'Device              : {DEVICE}')
log.info(f'Checkpoint out      : {CHECKPOINT_DIR}/phase3_v2.phase.pt')
log.info('Gate check every 1,000 steps — early stop when avg>90% min>70%')
log.info('')
log.info('KEY CHANGE vs prior runs (all stuck at 2-7%):')
log.info('  ROOT CAUSE: wave_loss (teacher-forced MSE) incompatible with gate (autoregressive)')
log.info('  FIX: REINFORCE — every 50 steps, roll out autoregressively, compute byte-accuracy')
log.info('       reward, do reward-weighted imitation update (advantage > 0 → reinforce,')
log.info('       advantage < 0 → suppress). Directly optimises what the gate measures.')
log.info('  Removed: manifold_loss and anchor_loss (flat gradients, both proved ineffective)')

TRAIN_METRICS = train(
    device=DEVICE,
    checkpoint_dir=Path(CHECKPOINT_DIR),
)

log.metric('final_gate_avg', f"{TRAIN_METRICS.get('final_gate_avg', 0):.1%}")
log.metric('final_gate_min', f"{TRAIN_METRICS.get('final_gate_min', 0):.1%}")
log.metric('best_step',      str(TRAIN_METRICS.get('best_step', 'N/A')))
log.metric('total_time_s',   f"{TRAIN_METRICS.get('total_time_s', 0):.0f}")

p3_local = os.path.join(CHECKPOINT_DIR, 'phase3_v2.phase.pt')
if os.path.exists(p3_local):
    size_mb = os.path.getsize(p3_local) / 1e6
    log.success(f'Checkpoint saved: phase3_v2.phase.pt ({size_mb:.1f} MB)')
else:
    log.error('Checkpoint NOT saved — check training script')

log.info('NOTE: Final in-memory gate may differ from checkpoint gate — trust Cell 6')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — TRAINING DIAGNOSTICS  (run immediately after Cell 5)    ║
# ╚══════════════════════════════════════════════════════════════════════╝
import json, math, torch
from pathlib import Path

# Ensure log is available even if Cell 3 was skipped / kernel restarted
try:
    log
except NameError:
    import sys
    sys.path.insert(0, str(Path('/content/FLUX')))
    from flux_utils import PhaseLogger
    log = PhaseLogger(phase=3)

log.separator('Cell 6 — Training Diagnostics')

diagnostics = {}
FAIL_COUNT  = 0
WARN_COUNT  = 0

def diag_check(name: str, passed: bool, is_fatal: bool = True, detail: str = ''):
    global FAIL_COUNT, WARN_COUNT
    tag   = 'FAIL' if (not passed and is_fatal) else ('WARN' if not passed else 'PASS')
    icon  = '✓' if passed else ('✗' if is_fatal else '⚠')
    msg   = f'{icon} [{tag}] {name}' + (f' — {detail}' if detail else '')
    print(msg)
    diagnostics[name] = {'passed': passed, 'detail': detail, 'fatal': is_fatal}
    if not passed and is_fatal:
        FAIL_COUNT += 1
    elif not passed:
        WARN_COUNT += 1

p3_ckpt_path = Path(CHECKPOINT_DIR) / 'phase3_v2.phase.pt'

# 1. Checkpoint file exists
diag_check('Checkpoint file exists', p3_ckpt_path.exists(),
           detail=str(p3_ckpt_path))

if p3_ckpt_path.exists():
    # 2. Loads without error
    try:
        ckpt = torch.load(p3_ckpt_path, map_location='cpu')
        diag_check('Checkpoint loads without error', True)
    except Exception as e:
        diag_check('Checkpoint loads without error', False, detail=str(e))
        ckpt = None

    if ckpt is not None:
        # 3. Required keys present
        required_keys = ['phase', 'config', 'state_dict', 'metrics']
        missing = [k for k in required_keys if k not in ckpt]
        diag_check('Required checkpoint keys present', len(missing) == 0,
                   detail=f'missing: {missing}' if missing else '')

        # 4. state_dict has generator key
        has_gen = 'generator' in ckpt.get('state_dict', {})
        diag_check('state_dict contains generator key', has_gen)

        # 5. Loss is finite
        metrics    = ckpt.get('metrics', {})
        final_loss = metrics.get('loss', metrics.get('final_loss', float('nan')))
        is_finite  = math.isfinite(final_loss) if final_loss == final_loss else False
        if final_loss == -1.0:
            diag_check('Loss is finite (no NaN/Inf)', True,
                       detail='loss guarded (-1.0 sentinel, NaN avoided)')
        else:
            diag_check('Loss is finite (no NaN/Inf)', is_finite,
                       detail=f'loss={final_loss}')

        # 6. Loss below threshold
        loss_ok = is_finite and 0 <= final_loss < 5.0
        diag_check('Loss below threshold (< 5.0)', loss_ok, is_fatal=False,
                   detail=f'loss={final_loss:.4f}' if is_finite else f'loss={final_loss}')

        # 7. Model loads state_dict
        try:
            sys.path.insert(0, V2_PHASE3)
            from wave_generator import WaveGenerator
            _gen = WaveGenerator()
            _gen.load_state_dict(ckpt['state_dict']['generator'])
            _gen.eval()
            diag_check('Generator loads state_dict', True)
        except Exception as e:
            diag_check('Generator loads state_dict', False, detail=str(e))
            _gen = None

        # 8. Forward pass produces correct shape
        if _gen is not None:
            try:
                with torch.no_grad():
                    dummy_ctx = torch.randn(512)
                    waves, _  = _gen.generate(field_context=dummy_ctx, max_waves=3)
                shape_ok = waves.shape[1] == 432
                diag_check('Wave shape correct [N, 432]', shape_ok,
                           detail=str(list(waves.shape)))

                # 9. Wave not degenerate
                std_ok = waves.std().item() > 1e-4
                diag_check('Wave not degenerate (std > 1e-4)', std_ok,
                           detail=f'std={waves.std().item():.6f}', is_fatal=False)

                # 10. Mini decode gate — WITH Phase 2 bridge (same path as training)
                from wave_to_text  import WaveToText
                from wave_to_field import WaveToField
                from field_to_wave import FieldToWave
                p1_c = torch.load(f'{CHECKPOINT_DIR}/phase1_v2.phase.pt', map_location='cpu')
                p2_c = torch.load(f'{CHECKPOINT_DIR}/phase2_v2.phase.pt', map_location='cpu')
                _wtt = WaveToText(wave_dim=432, hidden_dim=256, max_bytes=20)
                _wtt.load_state_dict(p1_c['state_dict']['wtt'])
                _wtt.eval()
                _w2f = WaveToField(wave_dim=432, field_dim=512)
                _w2f.load_state_dict(p2_c['state_dict']['bridge_wtf'])
                _w2f.eval()
                _f2w = FieldToWave(field_dim=512, wave_dim=432)
                _f2w.load_state_dict(p2_c['state_dict']['bridge_ftw'])
                _f2w.eval()
                with torch.no_grad():
                    # Snap waves onto CSE manifold through Phase 2 bridge
                    bridged = _f2w(_w2f(waves))
                    decoded_any = False
                    for i in range(bridged.shape[0]):
                        b = _wtt.decode(bridged[i])
                        if b and len(b) > 0:
                            decoded_any = True
                diag_check('Mini decode gate (bridged): at least some bytes decoded',
                           decoded_any, is_fatal=False)
            except Exception as e:
                diag_check('Forward pass runs without error', False, detail=str(e))

        # 11. Gate metrics
        gate_avg = metrics.get('decode_gate_avg', metrics.get('final_gate_avg', 0.0))
        gate_min = metrics.get('decode_gate_min', metrics.get('final_gate_min', 0.0))
        diag_check(f'Gate avg > 90% ({gate_avg:.1%})', gate_avg > 0.9, is_fatal=False,
                   detail=f'avg={gate_avg:.1%}')
        diag_check(f'Gate min > 70% ({gate_min:.1%})', gate_min > 0.7, is_fatal=False,
                   detail=f'min={gate_min:.1%}')

# Save diagnostics
diag_path = os.path.join(RESULTS_DIR, 'diagnostics.json')
with open(diag_path, 'w') as f:
    json.dump(diagnostics, f, indent=2)

print()
if FAIL_COUNT == 0:
    log.success(f'DIAGNOSTICS PASSED — {WARN_COUNT} warning(s), 0 fatal failures')
    log.success('Safe to proceed to Cell 7 (upload)')
else:
    log.error(f'DIAGNOSTICS FAILED — {FAIL_COUNT} fatal failure(s). DO NOT proceed.')
    log.error('Fix the issue and re-run Cell 5 + Cell 6 before uploading.')
    raise RuntimeError(f'{FAIL_COUNT} fatal diagnostic failure(s). See output above.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — UPLOAD TO HUGGINGFACE                                    ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os
from huggingface_hub import HfApi

log.separator('Cell 7 — Upload to HuggingFace')

# ── Re-resolve HF_TOKEN (guards against %reset -f clearing the variable)
_hf_token = HF_TOKEN if 'HF_TOKEN' in dir() else ''
_hf_token = _hf_token or os.environ.get('HF_TOKEN', '')

if not _hf_token:
    try:
        from google.colab import userdata
        _hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        pass

if not _hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = (UserSecretsClient().get_secret('HF_TOKEN') or '').strip()
    except Exception:
        pass

if not _hf_token:
    log.error('HF_TOKEN is empty — add it via Colab/Kaggle secrets and re-run Cell 3')
    raise ValueError('HF_TOKEN is required for upload. Run Cell 3 first.')

HF_TOKEN = _hf_token
os.environ['HF_TOKEN'] = _hf_token
log.success(f'HF_TOKEN resolved (len={len(_hf_token)})')

_api = HfApi(token=_hf_token)   # always pass _hf_token, never bare HF_TOKEN

# ── Upload checkpoint ────────────────────────────────────────────────
p3_local = os.path.join(CHECKPOINT_DIR, 'phase3_v2.phase.pt')
assert os.path.exists(p3_local), f'Checkpoint not found: {p3_local}'

size_mb = os.path.getsize(p3_local) / 1e6
log.info(f'Uploading phase3_v2.phase.pt ({size_mb:.1f} MB) → {HF_REPO_ID}')

_api.upload_file(
    path_or_fileobj=p3_local,
    path_in_repo='v2/phase3_v2.phase.pt',
    repo_id=HF_REPO_ID,
    repo_type='model',
    commit_message='Phase 3 v2: Wave Generator checkpoint',
)
log.success(f'Uploaded: {HF_REPO_ID}/v2/phase3_v2.phase.pt')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — TEST 1: Decode Gate Accuracy  (loads from HF)           ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil
from huggingface_hub import hf_hub_download

log.separator('Cell 8 — Test 1: Decode Gate Accuracy')

# Load checkpoint from HuggingFace (always use HF version for tests)
_cached = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename='v2/phase3_v2.phase.pt',
    token=_hf_token,
    force_download=True,
)
shutil.copy2(_cached, os.path.join(CHECKPOINT_DIR, 'phase3_v2.phase.pt'))
log.success('Checkpoint downloaded from HF for testing')

_env = {**os.environ, 'CHECKPOINT_DIR': CHECKPOINT_DIR}
result = subprocess.run(
    [sys.executable, f'{V2_PHASE3}/test_phase3_test1.py'],
    capture_output=True, text=True, cwd=REPO_PATH, env=_env,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    log.warning('Test 1 FAILED')
else:
    log.success('Test 1 PASSED — Decode Gate Accuracy')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — TEST 2: Context Sensitivity  (loads from HF)            ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys

log.separator('Cell 9 — Test 2: Context Sensitivity')

_env = {**os.environ, 'CHECKPOINT_DIR': CHECKPOINT_DIR}
result = subprocess.run(
    [sys.executable, f'{V2_PHASE3}/test_phase3_test2.py'],
    capture_output=True, text=True, cwd=REPO_PATH, env=_env,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    log.warning('Test 2 FAILED')
else:
    log.success('Test 2 PASSED — Context Sensitivity')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — TEST 3: Valid Word Rate  (loads from HF)               ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys

log.separator('Cell 10 — Test 3: Valid Word Rate')

_env = {**os.environ, 'CHECKPOINT_DIR': CHECKPOINT_DIR}
result = subprocess.run(
    [sys.executable, f'{V2_PHASE3}/test_phase3_test3.py'],
    capture_output=True, text=True, cwd=REPO_PATH, env=_env,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    log.warning('Test 3 FAILED')
else:
    log.success('Test 3 PASSED — Valid Word Rate')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — DEMO 1: Generation Pipeline  (loads from HF)           ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys

log.separator('Cell 11 — Demo 1: Generation Pipeline')

_env = {**os.environ, 'CHECKPOINT_DIR': CHECKPOINT_DIR}
result = subprocess.run(
    [sys.executable, f'{V2_PHASE3}/demo_phase3_demo1.py'],
    capture_output=True, text=True, cwd=REPO_PATH, env=_env,
)
print(result.stdout[:4000])
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    log.warning('Demo 1 FAILED')
else:
    log.success('Demo 1 PASSED')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — DEMO 2: Context Cluster Heatmap  (loads from HF)       ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

log.separator('Cell 12 — Demo 2: Context Cluster Heatmap')

_env = {**os.environ, 'CHECKPOINT_DIR': CHECKPOINT_DIR}
result = subprocess.run(
    [sys.executable, f'{V2_PHASE3}/demo_phase3_demo2.py'],
    capture_output=True, text=True, cwd=REPO_PATH, env=_env,
)
print(result.stdout[:2000])
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    log.warning('Demo 2 FAILED')
else:
    log.success('Demo 2 PASSED')

# Copy plot to RESULTS_DIR/graphs/ and display it
plot_src = Path(V2_PHASE3) / 'demo_phase3_demo2.png'
if plot_src.exists():
    plot_dst = Path(RESULTS_DIR) / 'graphs' / 'context_clusters.png'
    shutil.copy2(str(plot_src), str(plot_dst))
    img = mpimg.imread(str(plot_src))
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Phase 3 — Context-Dependent Generation Clusters')
    plt.tight_layout()
    plt.show()
    log.success(f'Plot saved to {plot_dst}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — SAVE RESULTS                                            ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, re, json, shutil, subprocess
from pathlib import Path
from huggingface_hub import HfApi

log.separator('Cell 13 — Save Results')

# ── 1. Collect log file ───────────────────────────────────────────────
log_src = Path(REPO_PATH) / 'logs' / 'phase3.log'
log_dst = Path(RESULTS_DIR) / 'logs' / 'phase3.log'
if log_src.exists():
    shutil.copy2(str(log_src), str(log_dst))
    log.success(f'Log copied → {log_dst}')

# ── 2. Collect training graphs from train script output ───────────────
for graph_name in ['training_loss.png', 'decode_gate.png', 'wave_similarity.png']:
    src = Path(V2_PHASE3) / graph_name
    if src.exists():
        shutil.copy2(str(src), str(Path(RESULTS_DIR) / 'graphs' / graph_name))
        log.success(f'Graph saved: {graph_name}')

# ── 3. Write metrics.json ─────────────────────────────────────────────
metrics_path = Path(RESULTS_DIR) / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(TRAIN_METRICS, f, indent=2)
log.success(f'metrics.json written')

# ── 4. Generate RESULTS_PHASE_3.md ───────────────────────────────────
results_md = Path(RESULTS_DIR) / 'RESULTS_PHASE_3.md'
lines = [
    '# FLUX v2 Phase 3 Results — Wave Generator',
    '',
    f'## Training Metrics',
    f'| Metric | Value |',
    f'|--------|-------|',
]
for k, v in TRAIN_METRICS.items():
    lines.append(f'| {k} | {v} |')
lines += ['', '## Files', f'- Checkpoint: `v2/phase3_v2.phase.pt` on `{HF_REPO_ID}`', '']
results_md.write_text('\n'.join(lines))
log.success('RESULTS_PHASE_3.md written')

# ── 5. Upload log to HuggingFace ──────────────────────────────────────
_api = HfApi(token=_hf_token)
if log_dst.exists():
    try:
        _api.upload_file(
            path_or_fileobj=str(log_dst),
            path_in_repo='v2/logs/phase3.log',
            repo_id=HF_REPO_ID,
            repo_type='model',
            commit_message='Phase 3 v2: training log',
        )
        log.success('Log uploaded to HuggingFace Hub')
    except Exception as e:
        log.warning(f'HF log upload failed: {e}')

# ── 6. Push to GitHub (v2 branch) ────────────────────────────────────
# Spec: scrub tokens → fetch → reset --soft → add → commit → push → finally cleanup
if GITHUB_TOKEN:
    TOKEN_PATTERN = re.compile(r'(hf_|ghp_)[A-Za-z0-9]+')

    # Scrub tokens from all files about to be committed
    for scrub_path in [log_dst, results_md, metrics_path]:
        if scrub_path.exists():
            raw  = scrub_path.read_text(errors='replace')
            clean = TOKEN_PATTERN.sub('[TOKEN_REDACTED]', raw)
            scrub_path.write_text(clean)
    log.info('Sensitive tokens scrubbed from result files')

    _clean_remote = f'https://github.com/Unseengap/FLUX.git'
    _auth_remote  = _clean_remote.replace('https://', f'https://Unseengap:{GITHUB_TOKEN}@')

    try:
        subprocess.run(['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin', _auth_remote],
                       check=True, capture_output=True)

        subprocess.run(['git', '-C', REPO_PATH, 'fetch', 'origin', BRANCH],
                       check=True, capture_output=True)
        subprocess.run(['git', '-C', REPO_PATH, 'reset', '--soft', f'origin/{BRANCH}'],
                       check=True, capture_output=True)

        files_to_add = [
            str(results_md),
            str(metrics_path),
            str(log_dst),
            str(Path(RESULTS_DIR) / 'diagnostics.json'),
            str(Path(RESULTS_DIR) / 'graphs' / 'context_clusters.png'),
        ]
        existing = [f for f in files_to_add if os.path.exists(f)]
        subprocess.run(['git', '-C', REPO_PATH, 'add'] + existing,
                       check=True, capture_output=True)

        status = subprocess.run(['git', '-C', REPO_PATH, 'status', '--porcelain'],
                                capture_output=True, text=True)
        if status.stdout.strip():
            subprocess.run(['git', '-C', REPO_PATH, 'commit',
                            '-m', 'Phase 3 v2: Wave Generator results [skip ci]'],
                           check=True, capture_output=True)
            subprocess.run(['git', '-C', REPO_PATH, 'push', 'origin', BRANCH],
                           check=True, capture_output=True)
            log.success(f'Results pushed to GitHub branch: {BRANCH}')
        else:
            log.info('No changes to commit')

    except subprocess.CalledProcessError as e:
        log.warning(f'GitHub push failed: {e.stderr}')
    finally:
        # Always restore clean remote URL — never leave token in git config
        subprocess.run(['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin', _clean_remote],
                       capture_output=True)
        log.info('Remote URL restored (token stripped)')
else:
    log.warning('No GITHUB_TOKEN — skipping GitHub push')

log.success('Cell 13 complete — all results saved')

# Phase 3 v2 — Final Summary

## What Phase 3 Trained
A **WaveGenerator** (GRU-based) that takes a field context vector (512-dim) and autoregressively
predicts the next semantic wave (432-dim). This is the first FLUX component that **generates** text
rather than merely encoding or storing it.

## Key Design Decisions
| Decision | Reason |
|----------|--------|
| Decode loss from step 1 | Legacy Phase 9.5 had no decode loss — root cause of all generation failures |
| Context decorrelation (L2-norm + LayerNorm + projection) | Prevents context collapse where all outputs collapse to cosine > 0.98 |
| Contrastive loss (weight=2.0) | Forces different prompts to produce distinguishable wave sequences |
| Scheduled sampling (0.5 → 0.9) | Prevents teacher-forcing dependency; model learns to track its own errors |
| Frozen CSE + WTT + W2F + F2W | Only WaveGenerator is trained; prior representations are preserved |

## Decode Gate Thresholds
- Early stop: **avg > 90%, min > 70%** across 8 diverse gate texts
- Test 1 pass: avg > 50%, min > 20% (generated vs original — harder than round-trip)

## Checkpoint
- HuggingFace: `UnseenGAP/FLUX` → `v2/phase3_v2.phase.pt`
- Contains: frozen CSE/WTT/W2F configs + WaveGenerator `state_dict`

## Next Phase
**Phase 4** — Wave Attention: learn which stored waves in the field are relevant to a query,
enabling FLUX to retrieve and condition generation on memory (O(log n) gravitational relevance).